# PCA Series | Chapter 0: Core PCA — The Full Theory

> **Chapter Goal:** Master PCA completely: understand its four equivalent mathematical interpretations, derive the algorithm from scratch, understand SVD's role, compute explained variance, and develop the ability to explain every step in an interview without hesitation.

**Prerequisites:** All `math/` chapters — especially eigen-decomposition, SVD, covariance matrix, and the Lagrangian derivation.

---

## 1. Why PCA? The Curse of Dimensionality

### **The Problem: High Dimensions Are Hostile**

As the number of features $d$ grows, several problems emerge:

**1. Distance Concentration:**
In high dimensions, the distance between any two random points becomes nearly the same. Consider $N$ random points uniformly distributed in a $d$-dimensional hypercube:
$$\frac{\text{max pairwise distance} - \text{min pairwise distance}}{\text{min pairwise distance}} \to 0 \quad \text{as } d \to \infty$$

Nearest-neighbor algorithms fail because "near" and "far" become meaningless. Distance-based models (KNN, kernel methods, clustering) degrade.

**2. Data Sparsity:**
The volume of a $d$-dimensional hypercube grows exponentially with $d$. To maintain the same data density, you need $N \propto e^d$ data points. In practice, data is always sparse in high dimensions — you're always extrapolating.

**3. Noise Amplification:**
More dimensions = more noise dimensions. If only 10 of 1000 features are informative, the 990 noise dimensions actively hurt your model. PCA can discard the noise dimensions.

**4. Computational Cost:**
Many algorithms scale as $O(d^2)$ or $O(d^3)$ — untenable for $d = 10^4$ or $d = 10^6$.

### **The JPEG Insight — Why Compression Works at All**
Real-world data is **not uniformly spread** across all dimensions. A 1000-pixel image has $10^{300}$ possible values if pixels were independent — but natural images occupy a tiny, low-dimensional manifold in this space. Pixels of the same object have similar values; adjacent pixels correlate; faces follow templates.

**PCA exploits structure:** it finds the low-dimensional sub-space where real data actually lives.

---

## 2. What PCA Does

**PCA finds a new coordinate system** — a set of orthogonal axes called **Principal Components** (PCs) — such that:
1. **PC1** points in the direction of maximum variance in the data.
2. **PC2** points in the direction of second-highest variance, **orthogonal** to PC1.
3. **PC $k$** points in the direction of $k$-th highest variance, orthogonal to all previous PCs.

Then we **project** data onto the top-$k$ PCs, discarding the remaining $d-k$ low-variance directions. The result is a compressed, decorrelated representation.

**Nothing is invented.** PCA only discovers structure already present in the data — it cannot add information that isn't there.

### **Summary in One Diagram**
```
 Original space (d=2):          PC space (k=1, compressed):

  y ↑   ╱‾‾╲                     PC1 ──────────────────────►
    |  ╱  ●● ╲                    ●●●●●●●●●●●●●●●●●●
    | ╱  ●●●  ╲                   (1D — most info kept)
    |╱   ●●   ╲
    └──────────── x
    Correlated 2D data            Data projected onto PC1
```

---

## 3. Interpretation 1: Maximum Variance

> **"PCA finds direction that maximizes the variance of projected data."**

### **Setup**
Let $X \in \mathbb{R}^{N \times d}$ be the **centered** data matrix (mean of each column = 0). We seek a unit vector $\vec{w} \in \mathbb{R}^d$ such that the projected scores $z_i = \vec{x}_i \cdot \vec{w}$ have maximum variance.

The sample variance of projections:
$$\text{Var}(\vec{w}) = \frac{1}{N-1} \sum_{i=1}^N z_i^2 = \frac{1}{N-1} \|X\vec{w}\|^2 = \frac{1}{N-1} \vec{w}^T X^T X \vec{w} = \vec{w}^T \Sigma \vec{w}$$

where $\Sigma = X^T X / (N-1)$ is the sample covariance matrix.

### **Optimization Problem**
$$\max_{\vec{w}} \vec{w}^T \Sigma \vec{w} \quad \text{subject to} \quad \|\vec{w}\|_2 = 1$$

### **Solution via Lagrangian** (Full Derivation)
Form: $\mathcal{L}(\vec{w}, \lambda) = \vec{w}^T \Sigma \vec{w} - \lambda(\vec{w}^T\vec{w} - 1)$

Stationarity: $\frac{\partial \mathcal{L}}{\partial \vec{w}} = 2\Sigma\vec{w} - 2\lambda\vec{w} = \vec{0}$

$$\Rightarrow \Sigma\vec{w} = \lambda\vec{w} \quad \text{(eigenvalue equation)}$$

Substituting back: $\text{Var}(\vec{w}) = \vec{w}^T \Sigma \vec{w} = \vec{w}^T (\lambda\vec{w}) = \lambda$

> **The variance achieved equals the eigenvalue.** To maximize variance → choose $\vec{w}_1$ = eigenvector of $\Sigma$ with largest eigenvalue $\lambda_1$.

For PC $k$: add the orthogonality constraint $\vec{w}_k \perp \vec{w}_1, ..., \vec{w}_{k-1}$. The solution is the eigenvector with the $k$-th largest eigenvalue $\lambda_k$.

---

## 4. Interpretation 2: Minimum Reconstruction Error

> **"PCA finds the subspace that loses the least information when data is projected onto it."**

### **Setup**
Project each centered data point $\vec{x}_i$ onto a $k$-dimensional subspace spanned by orthonormal vectors $\{\vec{w}_1, ..., \vec{w}_k\}$. The reconstruction:
$$\hat{\vec{x}}_i = \sum_{j=1}^k (\vec{x}_i \cdot \vec{w}_j) \vec{w}_j = W_k W_k^T \vec{x}_i$$

The reconstruction error (total over all data points):
$$\mathcal{E}(W_k) = \sum_{i=1}^N \|\vec{x}_i - \hat{\vec{x}}_i\|^2 = \|X - XW_k W_k^T\|_F^2$$

### **Equivalence to Interpretation 1**
Using the Pythagorean theorem (since $\vec{x}_i - \hat{\vec{x}}_i \perp \hat{\vec{x}}_i$):
$$\|\vec{x}_i\|^2 = \|\hat{\vec{x}}_i\|^2 + \|\vec{x}_i - \hat{\vec{x}}_i\|^2$$

Summing over all $i$ and all directions:
$$\underbrace{\sum_i \|\vec{x}_i\|^2}_{\text{total variance (fixed)}} = \underbrace{\sum_i \|\hat{\vec{x}}_i\|^2}_{\text{captured variance}} + \underbrace{\mathcal{E}(W_k)}_{\text{reconstruction error}}$$

Since total variance is fixed: **minimizing reconstruction error = maximizing captured variance.** Both interpretations are mathematically equivalent.

### **The Minimum Error**
The minimum reconstruction error, achieved when $W_k$ contains the top-$k$ eigenvectors:
$$\mathcal{E}^* = \sum_{j=k+1}^d \lambda_j = \text{tr}(\Sigma) - \sum_{j=1}^k \lambda_j$$

The error is exactly the sum of the **discarded eigenvalues** — the variance in the directions we threw away.

---

## 5. Interpretation 3: Decorrelation

> **"PCA diagonalizes the covariance matrix — making all pairs of PCs uncorrelated."**

### **The Statement**
Let $W = [\vec{w}_1 | \cdots | \vec{w}_d]$ be the matrix of all eigenvectors of $\Sigma$ (columns). Since $\Sigma$ is symmetric, $W$ is orthogonal: $W^T W = I$.

The PC scores matrix: $Z = XW \in \mathbb{R}^{N \times d}$.

The covariance matrix of $Z$:
$$\Sigma_Z = \frac{1}{N-1} Z^T Z = \frac{1}{N-1} (XW)^T (XW) = W^T \underbrace{\frac{X^T X}{N-1}}_{\Sigma} W = W^T \Sigma W$$

Since $\Sigma W = W \Lambda$ (eigendecomposition):
$$\Sigma_Z = W^T W \Lambda = \Lambda = \text{diag}(\lambda_1, ..., \lambda_d)$$

**The covariance matrix of the PC scores is diagonal.** This means:
- $\text{Var}(Z_j) = \lambda_j$ (variance of PC $j$ = eigenvalue $j$) ✅
- $\text{Cov}(Z_j, Z_k) = 0$ for $j \neq k$ (all PCs are uncorrelated) ✅

### **Geometric Meaning**
PCA applies a rigid rotation (matrix $W$) to the data. This rotation aligns the coordinate axes with the ellipse axes of the data cloud. In PC space, the data ellipse is axis-aligned — its covariance matrix is diagonal.

For Gaussian data, uncorrelated = independent. PCA produces statistically independent components when data is Gaussian.

---

## 6. Interpretation 4: Best Low-Rank Approximation (SVD / Eckart-Young)

> **"PCA gives the best possible rank-$k$ linear approximation to the data matrix."**

### **The Eckart-Young Theorem**
Let $X = U\Sigma_s V^T$ be the SVD of the centered data matrix $X$. The truncated SVD:
$$X_k = U_k \Sigma_{s,k} V_k^T = \sum_{i=1}^k \sigma_i \vec{u}_i \vec{v}_i^T$$

is the best rank-$k$ approximation to $X$ in the Frobenius norm:
$$X_k = \arg\min_{\text{rank}(M)=k} \|X - M\|_F^2 \quad \text{with error} \quad \|X - X_k\|_F^2 = \sum_{i=k+1}^{r} \sigma_i^2$$

### **The SVD-PCA Correspondence**

| PCA Language | SVD of $X$ |
| :--- | :--- |
| PC directions (loadings) | Right singular vectors: columns of $V$ |
| PC scores | $Z = U\Sigma_s$ (same as $XV$) |
| Variance of PC $k$ | $\sigma_k^2 / (N-1)$ |
| Reconstruction | $\hat{X} = X_k = U_k \Sigma_{s,k} V_k^T$ |
| Reconstruction error | $\sum_{i>k} \sigma_i^2 / (N-1)$ |

### **Why This Matters**
This interpretation tells us that among **all** possible ways to reduce the dimensionality of the data (any linear mapping), PCA is the **optimal** choice — it loses the least information. No other linear method can do better in terms of reconstruction quality. This is a strong theoretical guarantee.

---

## 7. The PCA Algorithm: Step by Step

### **Input:** Data matrix $X \in \mathbb{R}^{N \times d}$, number of components $k$.

**Step 1: Center the data**
$$\vec{\mu} = \frac{1}{N} \sum_{i=1}^N \vec{x}_i, \quad X_{\text{centered}} = X - \mathbf{1}\vec{\mu}^T$$

> Why centering? The covariance matrix is defined for centered data. Without centering, the first PC would point toward the data mean, not the direction of variance.

**Step 2: (Optional but critical) Standardize**
$$X_{\text{std}} = X_{\text{centered}} \oslash \vec{\sigma}$$
where $\vec{\sigma}$ is the vector of feature standard deviations (element-wise division). See Statistics Ch0 for why this is essential when features are in different units.

**Step 3: Compute the covariance matrix** (or use SVD directly)
$$\Sigma = \frac{1}{N-1} X_{\text{std}}^T X_{\text{std}} \in \mathbb{R}^{d \times d}$$

**Step 4: Eigendecompose (or SVD)**
Option A: $\Sigma = Q\Lambda Q^T$ — eigendecompose, sort by $\lambda$ descending.
Option B: $X = U\Sigma_s V^T$ — SVD directly (preferred: numerically stable, no need for Step 3).

**Step 5: Select top-$k$ components**
$$W_k = [\vec{w}_1 | \vec{w}_2 | \cdots | \vec{w}_k] \in \mathbb{R}^{d \times k}$$

**Step 6: Project data**
$$Z = X_{\text{std}} W_k \in \mathbb{R}^{N \times k}$$

$Z$ is the **low-dimensional representation** — each row is a data point expressed in the PC coordinate system.

**Step 7: (Optional) Reconstruct**
$$\hat{X}_{\text{std}} = Z W_k^T = X_{\text{std}} W_k W_k^T$$
$$\hat{X} = \hat{X}_{\text{std}} \odot \vec{\sigma} + \vec{\mu}$$

---

## 8. Explained Variance Ratio & Choosing $k$

### **Explained Variance Ratio (EVR)**
$$\text{EVR}_k = \frac{\lambda_k}{\sum_{i=1}^d \lambda_i} = \frac{\sigma_k^2}{\sum_i \sigma_i^2}$$

EVR tells you what fraction of total variance is captured by PC $k$ alone.

**Cumulative EVR:** $\sum_{j=1}^k \text{EVR}_j$ — fraction of variance captured by first $k$ PCs.

### **Methods for Choosing $k$**

| Method | Rule | Pro | Con |
| :--- | :--- | :--- | :--- |
| **Threshold** | Keep enough PCs until cumulative EVR $\geq 95\%$ | Simple, interpretable | Threshold is arbitrary |
| **Elbow (Scree plot)** | Plot EVR vs $k$; look for bend | Visual, intuitive | Bend often ambiguous |
| **Kaiser rule** | Keep PCs with $\lambda_k > 1$ (for standardized data) | Principled for correl. matrices | Doesn't generalize |
| **Marchenko-Pastur** | Keep PCs with $\lambda_k > \lambda_+$ (noise threshold) | Statistically motivated | Requires $N, d$ both large |
| **Cross-validation** | Choose $k$ that minimizes validation reconstruction error | Data-driven | Expensive |
| **Task-driven** | Choose $k$ that maximizes downstream model performance | Directly optimizes goal | Dependent on task |

> **Interview tip:** "The right $k$ depends on your goal. For visualization, $k=2$ or $k=3$. For preprocessing, use the 95% variance rule or cross-validation. For noise removal, use Marchenko-Pastur."

---

## 9. ✍️ Worked Example: Manual PCA on 4-Feature Dataset

Consider $N=4$ data points with $d=2$ features (for illustration):
$$X = \begin{bmatrix} 2 & 1 \\ 4 & 3 \\ 6 & 5 \\ 8 & 7 \end{bmatrix}$$

**Step 1: Center**
$$\vec{\mu} = [5, 4], \quad X_c = \begin{bmatrix} -3 & -3 \\ -1 & -1 \\ 1 & 1 \\ 3 & 3 \end{bmatrix}$$

**Step 2: Covariance matrix**
$$\Sigma = \frac{1}{3} X_c^T X_c = \frac{1}{3} \begin{bmatrix} 20 & 20 \\ 20 & 20 \end{bmatrix} = \begin{bmatrix} 6.67 & 6.67 \\ 6.67 & 6.67 \end{bmatrix}$$

**Step 3: Eigenvalues**
$$\det(\Sigma - \lambda I) = (6.67-\lambda)^2 - 6.67^2 = 0 \Rightarrow \lambda(\lambda - 13.33) = 0$$
$$\lambda_1 = 13.33, \quad \lambda_2 = 0$$

$\lambda_2 = 0$ confirms all data lies on a 1D line (perfectly correlated features). One PC captures 100% of variance.

**Step 4: Eigenvector for $\lambda_1 = 13.33$**
$$(\Sigma - 13.33I)\vec{w} = 0 \Rightarrow \vec{w}_1 = \frac{1}{\sqrt{2}}[1, 1]^T$$

**Step 5: Project**
$$Z = X_c \vec{w}_1 = \frac{1}{\sqrt{2}} \begin{bmatrix} -3+(-3) \\ -1+(-1) \\ 1+1 \\ 3+3 \end{bmatrix} = \frac{1}{\sqrt{2}}[-6, -2, 2, 6]^T \approx [-4.24, -1.41, 1.41, 4.24]^T$$

**Verify:** Variance of $Z = \frac{1}{3}(17.97 + 2 + 2 + 17.97) = 13.33 = \lambda_1$ ✅

---

## 10. Pros, Cons & Edge Cases

### **Pros**
- ✅ Optimal linear compression — no linear method does better (Eckart-Young theorem)
- ✅ Removes correlations completely — PCs are always uncorrelated
- ✅ Fast: $O(Nd \min(N,d))$ via thin SVD
- ✅ Deterministic — always gives the same result
- ✅ No hyperparameters (except $k$)

### **Cons / Limitations**
- ❌ **Linear only** — cannot capture non-linear manifolds (use Kernel PCA)
- ❌ **Unsupervised** — ignores labels; high-variance direction may not be the most discriminative (use LDA)
- ❌ **Variance ≠ importance** — an irrelevant feature with large variance will dominate
- ❌ **Not robust** — one gross outlier can completely rotate PCs (use Robust PCA)
- ❌ **PC axes are uninterpretable** — each PC is a mix of all original features
- ❌ **Must standardize** — sensitive to feature scales

### **Edge Cases**

**Identical eigenvalues:** If $\lambda_i = \lambda_j$, the corresponding eigenspace is a plane — any two orthonormal vectors in that plane work. PCA has a **rotational ambiguity** within degenerate eigenspaces. This can cause instability in the PC directions (a small perturbation to the data changes the direction dramatically). Solution: be cautious interpreting individual PCs when eigenvalues are close.

**Sign ambiguity:** For any eigenvector $\vec{w}$, $-\vec{w}$ is also a valid eigenvector with the same eigenvalue. PCA can flip the sign of PCs between runs. This doesn't affect reconstruction or variance, but makes direct comparison between runs unreliable. sklearn handles this by fixing the sign of the largest coefficient in each PC.

**$N < d$ (high-dimensional):** Covariance matrix is rank-deficient — at most $N-1$ non-zero PCs. Cannot run standard eigendecomposition on $\Sigma$ — use SVD of $X$ directly or Dual PCA. See Chapter 2.

**All features independent:** If $\Sigma$ is already diagonal, all PCs = original feature axes. PCA rotation = identity. PCA doesn't help for compression in this case.

---

## 11. Interview Q&A

**Q1: What does PCA do, in one sentence?**
> PCA finds an orthogonal rotation of the feature space such that the resulting axes capture maximum variance in decreasing order.

**Q2: Derive PCA from first principles.**
> We want to maximize $\vec{w}^T \Sigma \vec{w}$ subject to $\|\vec{w}\| = 1$. Forming the Lagrangian and differentiating gives $\Sigma \vec{w} = \lambda \vec{w}$ — an eigenvalue equation. The achieved variance equals $\lambda$, so we choose the eigenvector of $\Sigma$ with the largest eigenvalue. Subsequent PCs are eigenvectors with the next largest eigenvalues.

**Q3: Why must you standardize before PCA?**
> Because PCA maximizes variance, features with large scales (e.g., salary in dollars) will dominate over features with small scales (e.g., age in years). Standardization equalizes scales so that PCA responds to correlation structure rather than unit choice.

**Q4: Why does sklearn use SVD instead of eigendecomposition?**
> Two reasons: (1) numerical stability — computing $\Sigma = X^T X$ squares the condition number of the problem; SVD works directly on $X$ and is more numerically stable. (2) Generality — SVD works for any shaped matrix, including $N < d$ cases where $\Sigma$ is singular.

**Q5: What does the explained variance ratio tell you?**
> EVR$_k = \lambda_k / \sum \lambda_i$ is the fraction of total data variance captured by PC $k$. Cumulative EVR tells you what fraction of variance is retained by the first $k$ PCs. It does NOT tell you prediction accuracy on any downstream task.

**Q6: Can two different datasets have the same PCA result?**
> Yes. PCA only depends on the covariance matrix $\Sigma$. Two datasets with the same covariance matrix but different higher-order statistics (e.g., one Gaussian, one uniform) produce identical PCA results.

**Q7: What happens if eigenvalues are all equal?**
> If $\Sigma = c \cdot I$ (isotropic), all eigenvalues equal $c$. Every direction has the same variance. PCA cannot prefer any rotation — all axes are equally valid PCs. This is the worst case for PCA: data with no correlation structure.

**Q8: What is the relationship between PCA and the covariance matrix?**
> PCA performs the eigendecomposition of the covariance matrix: $\Sigma = Q\Lambda Q^T$. The columns of $Q$ are the principal components; the diagonal of $\Lambda$ are the variances. PCA is entirely defined by the covariance matrix.

**Q9: Is PCA sensitive to outliers?**
> Yes, severely. A single gross outlier can inflate a particular direction's variance enormously and rotate the first PC toward the outlier. Solution: use Robust PCA (L+S decomposition) or remove outliers before running PCA.

**Q10: How is PCA different from feature selection?**
> Feature selection picks a subset of original features. PCA creates entirely new features (the PCs) which are linear combinations of all original features. PCA often captures more information with fewer dimensions but produces uninterpretable features. Use PCA for compression; use feature selection for interpretability.

---

## 12. Implementation Playground

In [ ]:
# ─── CELL 1: PCA From Scratch — Manual Implementation ─────────────────────────
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris

# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target  # 150 x 4
feature_names = iris.feature_names

print(f"Dataset shape: {X.shape}  ({X.shape[0]} samples, {X.shape[1]} features)")

# ── My PCA (from scratch) ─────────────────────────────────────────────────────
class ManualPCA:
    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        # Step 1: Center
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_

        # Step 2: SVD (more numerically stable than cov eigendecomp)
        U, s, Vt = np.linalg.svd(Xc, full_matrices=False)

        # Step 3: Sort (SVD already gives descending singular values)
        self.components_ = Vt[:self.n_components]         # k x d
        self.explained_variance_ = (s[:self.n_components]**2) / (X.shape[0] - 1)
        self.explained_variance_ratio_ = (s**2 / (s**2).sum())[:self.n_components]
        self.singular_values_ = s[:self.n_components]
        return self

    def transform(self, X):
        return (X - self.mean_) @ self.components_.T       # N x k

    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_           # N x d

# Standardize and apply PCA
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

k = 4
my_pca = ManualPCA(n_components=k).fit(X_std)
sk_pca = PCA(n_components=k).fit(X_std)

print("\n=== Manual vs Sklearn PCA ===")
print(f"Explained variance ratios:")
print(f"  Manual:  {my_pca.explained_variance_ratio_.round(4)}")
print(f"  Sklearn: {sk_pca.explained_variance_ratio_.round(4)}")
print(f"  Match: {np.allclose(my_pca.explained_variance_ratio_, sk_pca.explained_variance_ratio_)}")

# Transform and verify
Z_my = my_pca.transform(X_std)
Z_sk = sk_pca.transform(X_std)
# May differ in sign — check absolute differences after sign alignment
signs = np.sign((Z_my * Z_sk).sum(axis=0))
print(f"\nScores match (up to sign flip)? {np.allclose(Z_my, Z_sk * signs, atol=1e-6)}")

In [ ]:
# ─── CELL 2: Scree Plot + Cumulative Explained Variance ───────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Digits dataset (1797 samples x 64 features)
X_digits, y_digits = load_digits(return_X_y=True)
X_std = StandardScaler().fit_transform(X_digits)

pca_full = PCA().fit(X_std)  # All components

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Scree plot
axes[0].bar(range(1, 21), pca_full.explained_variance_ratio_[:20] * 100,
            color='steelblue', alpha=0.8)
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('% Variance Explained')
axes[0].set_title('Scree Plot: Digits Dataset\n(Individual Variance per PC)', fontsize=11)
axes[0].grid(alpha=0.3, axis='y')

# Cumulative variance
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
axes[1].plot(range(1, len(cumvar)+1), cumvar, color='darkorange', lw=2)
for threshold in [80, 90, 95, 99]:
    k_needed = np.argmax(cumvar >= threshold) + 1
    axes[1].axhline(threshold, color='gray', ls=':', alpha=0.6)
    axes[1].annotate(f'{threshold}%: k={k_needed}',
                     xy=(k_needed, threshold), xytext=(k_needed+2, threshold-3), fontsize=8)
axes[1].axhline(95, color='red', ls='--', lw=1.5)
axes[1].set_xlabel('Number of Components k'); axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Explained Variance', fontsize=11)
axes[1].grid(alpha=0.3); axes[1].set_xlim(1, 64)

# Reconstruction quality at different k
ks = [5, 15, 30, 50, 64]
errors = []
for k in ks:
    pca_k = PCA(n_components=k)
    Z = pca_k.fit_transform(X_std)
    X_recon = pca_k.inverse_transform(Z)
    err = np.mean((X_std - X_recon)**2)
    errors.append(err)

axes[2].plot(ks, errors, 'o-', color='forestgreen', lw=2, markersize=8)
axes[2].set_xlabel('Number of Components k'); axes[2].set_ylabel('Mean Reconstruction Error (MSE)')
axes[2].set_title('Reconstruction Error vs k', fontsize=11)
axes[2].grid(alpha=0.3)

plt.suptitle('PCA: Explained Variance & Reconstruction — Digits Dataset (64 features → k PCs)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 3: Numerically Verify All 4 Interpretations ─────────────────────────
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X, y = load_breast_cancer(return_X_y=True)  # 569 x 30
X_std = StandardScaler().fit_transform(X)
N, d = X_std.shape
k = 5

pca = PCA(n_components=k).fit(X_std)
Z = pca.transform(X_std)
W_k = pca.components_.T  # d x k

print("=" * 55)
print("Verifying All 4 PCA Interpretations on Breast Cancer Data")
print("=" * 55)

# Interpretation 1: Maximum variance
print("\n1. Maximum Variance:")
Sigma = X_std.T @ X_std / (N - 1)
for i in range(k):
    w = W_k[:, i]
    var_achieved = w @ Sigma @ w
    eigenval = pca.explained_variance_[i]
    print(f"   PC{i+1}: wᵀΣw = {var_achieved:.4f}  = λ{i+1} = {eigenval:.4f}  ✓ {np.isclose(var_achieved, eigenval)}")

# Interpretation 2: Minimum reconstruction error
print("\n2. Minimum Reconstruction Error:")
X_recon = pca.inverse_transform(Z)
recon_err = np.sum((X_std - X_recon)**2) / (N-1)
# Predicted by discarded eigenvalues
pca_full = PCA().fit(X_std)
predicted_err = pca_full.explained_variance_[k:].sum()
print(f"   Actual MSE per feature: {recon_err:.4f}")
print(f"   Sum of discarded λ:     {predicted_err:.4f}  ≈ match ✓")

# Interpretation 3: Decorrelation
print("\n3. Decorrelation:")
cov_Z = Z.T @ Z / (N - 1)  # should be diagonal = Lambda
off_diag_max = np.max(np.abs(cov_Z - np.diag(np.diag(cov_Z))))
print(f"   Max off-diagonal element of cov(Z): {off_diag_max:.2e}  (≈ 0 ✓)")
print(f"   Diagonal of cov(Z): {np.diag(cov_Z).round(4)}")
print(f"   Eigenvalues:        {pca.explained_variance_.round(4)}")

# Interpretation 4: Best low-rank approximation (Eckart-Young)
print("\n4. Best Low-Rank Approximation (Eckart-Young):")
U, s, Vt = np.linalg.svd(X_std, full_matrices=False)
X_k_svd = (U[:, :k] * s[:k]) @ Vt[:k, :]  # rank-k truncated SVD
svd_error = np.linalg.norm(X_std - X_k_svd, 'fro')**2
pca_error = np.linalg.norm(X_std - X_recon, 'fro')**2
print(f"   Truncated SVD Frobenius error²: {svd_error:.4f}")
print(f"   PCA reconstruction error²:      {pca_error:.4f}")
print(f"   Are they equal? {np.isclose(svd_error, pca_error)}  ✓")

In [ ]:
# ─── CELL 4: PCA Visualization — 2D & 3D Projections ─────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

datasets = [
    (load_iris(),    'Iris (4 → 2 PCs)',    load_iris().target_names),
    (load_wine(),    'Wine (13 → 2 PCs)',   load_wine().target_names),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']

for ax, (data, title, class_names) in zip(axes, datasets):
    X_std = StandardScaler().fit_transform(data.data)
    pca = PCA(n_components=2)
    Z = pca.fit_transform(X_std)
    y = data.target

    for cls_idx, cls_name in enumerate(class_names):
        mask = y == cls_idx
        ax.scatter(Z[mask, 0], Z[mask, 1], s=20, alpha=0.7,
                   color=colors[cls_idx], label=cls_name)

    evr = pca.explained_variance_ratio_
    ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}% variance)')
    ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}% variance)')
    ax.set_title(f'{title}\nTotal: {evr.sum()*100:.1f}% variance retained', fontsize=11)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('PCA for Visualization: Projecting to 2D', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()